In [1]:
import os

input_dir = os.path.abspath("inputs/docking/carA_homologs_2C")
output_dir = os.path.abspath("outputs/docking/carA_homologs_2C")
ligand_pdbqt = os.path.abspath("inputs/rosetta/subs/adi.pdbqt")
box_path = os.path.join(input_dir, "box_coords.box.txt")
sh_path = os.path.join(input_dir, "run_vina.sh")

with open(sh_path, 'w') as f:
    f.write("#!/bin/bash\n")
    f.write("set -e\n\n")

    for entry in sorted(os.listdir(input_dir)):
        entry_dir = os.path.join(input_dir, entry)
        if not os.path.isdir(entry_dir):
            continue

        out_dir = os.path.join(output_dir, entry)
        f.write(f"mkdir -p {out_dir}\n")
        f.write(f"cd {entry_dir}\n")

        for pdb_name in sorted(os.listdir(entry_dir)):
            if not pdb_name.endswith('.pdb'):
                continue
            base = pdb_name.replace('.pdb', '')

            # Remove ADI HETATM lines in-place (keep AMP)
            f.write(f"grep -v 'ADI' {pdb_name} > {pdb_name}.tmp && mv {pdb_name}.tmp {pdb_name}\n")
            # Convert pdb to pdbqt (suppress obabel warnings on stderr)
            f.write(f"obabel {pdb_name} -O {base}.pdbqt -xr -xh 2>/dev/null\n")
            # Dock (vina 1.2.x has no --log; redirect stdout to log file)
            f.write(
                f"vina --receptor {base}.pdbqt --ligand {ligand_pdbqt} "
                f"--config {box_path} --cpu 8 "
                f"--out {out_dir}/ligand_{base}.pdbqt > {out_dir}/{base}.log 2>&1\n\n"
            )

In [ ]:
import os

input_dir = os.path.abspath("inputs/docking/carA_homologs_2C")
output_dir = os.path.abspath("outputs/docking/carA_homologs_2C")
ligand_pdbqt = os.path.abspath("inputs/rosetta/subs/adi.pdbqt")
box_path = os.path.join(input_dir, "box_coords.box.txt")
sh_path = os.path.join(input_dir, "run_vina.sh")

# Collect entries (directories) up front so we know the total for progress logging
entries = [
    e for e in sorted(os.listdir(input_dir))
    if os.path.isdir(os.path.join(input_dir, e))
]
total = len(entries)

with open(sh_path, 'w') as f:
    f.write("#!/bin/bash\n")
    f.write("set -e\n\n")

    # --- progress logging setup ---
    f.write(f"TOTAL={total}\n")
    f.write("START=$SECONDS\n")
    f.write("i=0\n")
    # format seconds as HH:MM:SS
    f.write("fmt(){ printf '%02d:%02d:%02d' $(($1/3600)) $(($1%3600/60)) $(($1%60)); }\n\n")

    for entry in entries:
        entry_dir = os.path.join(input_dir, entry)
        out_dir = os.path.join(output_dir, entry)

        # progress line for this entry
        f.write("i=$((i+1))\n")
        f.write(f'echo "[$i/$TOTAL] docking {entry} (elapsed $(fmt $((SECONDS-START))))"\n')

        f.write(f"mkdir -p {out_dir}\n")
        f.write(f"cd {entry_dir}\n")

        for pdb_name in sorted(os.listdir(entry_dir)):
            if not pdb_name.endswith('.pdb'):
                continue
            base = pdb_name.replace('.pdb', '')

            # Remove ADI HETATM lines in-place (keep AMP)
            f.write(f"grep -v 'ADI' {pdb_name} > {pdb_name}.tmp && mv {pdb_name}.tmp {pdb_name}\n")
            # Convert pdb to pdbqt (suppress obabel warnings on stderr)
            f.write(f"obabel {pdb_name} -O {base}.pdbqt -xr -xh 2>/dev/null\n")
            # Dock (vina 1.2.x has no --log; redirect stdout to log file)
            f.write(
                f"vina --receptor {base}.pdbqt --ligand {ligand_pdbqt} "
                f"--config {box_path} --cpu 8 "
                f"--out {out_dir}/ligand_{base}.pdbqt > {out_dir}/{base}.log 2>&1\n"
            )

        # done with this entry: report elapsed + ETA for the rest
        f.write("elapsed=$((SECONDS-START))\n")
        f.write("remaining=$(( i>0 ? (TOTAL-i)*elapsed/i : 0 ))\n")
        f.write('echo "    done $i/$TOTAL | elapsed $(fmt $elapsed) | ETA $(fmt $remaining)"\n\n')

    f.write('echo "all done in $(fmt $((SECONDS-START)))"\n')